# 08 · End-to-end demo: AOI + raster -> NDVI -> crop health -> stored result

This is the whole first-experience pipeline in one notebook, using the engines
directly and saving the outcome exactly like `POST /api/ndvi` does:

```
load AOI + raster -> clip + band stack -> NDVI -> statistics
     -> crop-health classes + weak patches -> visuals -> stored result
```

Swap `demo_scene()` for your own GeoTIFF + GeoJSON and nothing else changes.

In [ ]:
%matplotlib inline
# --- Vibes Lite bootstrap -----------------------------------------------------
# These notebooks drive the processing engines directly (no HTTP layer needed).
import sys
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

import config, data, geospatial, preprocessing, indices, analysis
import opencv as vl_cv          # Vibes Lite's own engine (it wraps cv2 internally)
import segmentation, timeseries, visualization, storage

PATHS = config.ensure_directories()
NOTEBOOK_OUT = PATHS["outputs"] / "notebooks"
NOTEBOOK_OUT.mkdir(parents=True, exist_ok=True)

print("project root :", ROOT)
print("raw data     :", PATHS["raw_data"].relative_to(ROOT))
print("notebook out :", NOTEBOOK_OUT.relative_to(ROOT))

In [ ]:
DEMO_NAME = "synthetic_fields"


def demo_scene(name: str = DEMO_NAME, size: int = 900):
    """Return (raster, aoi, csv) for the labelled synthetic demo scene.

    The scene is only generated when missing. It is explicitly synthetic - its
    GeoTIFF tags carry SYNTHETIC=true - so nothing here pretends to be a real
    satellite acquisition.
    """
    raster = PATHS["raw_data"] / f"{name}.tif"
    aoi = PATHS["raw_data"] / f"{name}.geojson"
    table = PATHS["raw_data"] / f"{name}_ndvi_history.csv"
    if not raster.exists() or not aoi.exists():
        scene = data.generate_synthetic_scene(name=name, size=size, overwrite=True)
        return scene["paths"]["raster"], scene["paths"]["aoi"], scene["paths"]["table"]
    return raster, aoi, table


RASTER, AOI_FILE, HISTORY_CSV = demo_scene()
print("raster :", RASTER.name)
print("aoi    :", AOI_FILE.name)
print("history:", HISTORY_CSV.name)

## Step 1 · AOI and raster

In [ ]:
metadata = data.get_raster_metadata(RASTER)
validation = data.validate_raster(RASTER)
aoi = data.load_aoi(AOI_FILE)
resolution = indices.resolve_band_map(metadata, None)
thresholds = config.health_thresholds()

print("dataset    :", metadata["name"], "|", metadata["band_count"], "bands |", metadata["crs"]["name"])
print("synthetic  :", metadata["synthetic"], "-", "labelled demo scene" if metadata["synthetic"] else "real input")
print("field      :", aoi["name"], "|", round(aoi["area_ha"], 2), "ha |", aoi["feature_count"], "features")
print("bands used :", resolution["bands"])
print("validation :", validation["warnings"] or "clean")

## Step 2 · Clip to the field, compute NDVI

In [ ]:
clipped = geospatial.clip_raster_to_aoi(
    str(RASTER), aoi["gdf"], bands=sorted(set(resolution["bands"].values())), max_pixels=config.max_analysis_pixels()
)
stack = preprocessing.band_stack(clipped, resolution["bands"])
valid = np.asarray(stack["valid_mask"])

ndvi = np.asarray(indices.compute_index("NDVI", stack["bands"], valid=valid)["array"], dtype="float32")
statistics = indices.index_statistics(ndvi, valid=valid)

print("window        :", stack["shape"], "| pixel area:", stack["pixel_area_ha"], "ha")
print("valid pixels  :", statistics["valid_pixels"], "of", int(valid.size))
print("NDVI min/mean/median/max: %.3f / %.3f / %.3f / %.3f" % (
    statistics["min"], statistics["mean"], statistics["median"], statistics["max"]))

## Step 3 · Crop health, weak patches, anomalies

In [ ]:
classes = analysis.classify_health(ndvi, thresholds=thresholds, valid=valid)
health = analysis.health_statistics(
    classes, pixel_area_ha=stack["pixel_area_ha"], thresholds=thresholds, field_area_ha=aoi["area_ha"]
)
zones = analysis.health_zones(classes, stack["transform"], stack["crs"], min_area_ha=0.05, pixel_area_ha=stack["pixel_area_ha"])
weak = analysis.detect_weak_vegetation(
    ndvi, stack["transform"], stack["crs"], thresholds=thresholds, valid=valid,
    min_area_ha=0.05, pixel_area_ha=stack["pixel_area_ha"],
)
anomalies = analysis.detect_anomalies(ndvi, valid=valid, method="zscore", threshold=2.0)

summary = analysis.field_summary(
    index_name="NDVI", statistics=statistics, health=health, weak_vegetation=weak, anomalies=anomalies,
    field_area_ha=aoi["area_ha"], analysed_area_ha=health["analysed_area_ha"], thresholds=thresholds,
)
summary["largest_patch_ha"] = weak.get("largest_patch_ha")
interpretation = analysis.interpret(summary)

print("health classes :", {item["key"]: item["percentage"] for item in health["classes"]})
print("healthy / weak :", health["healthy_pct"], "% /", health["weak_pct"], "%")
print("zones          :", zones["total_zones"], "| weak patches:", weak["patch_count"], "| anomalies:", anomalies.get("count"))
print("verdict        :", interpretation["verdict"])

## Step 4 · Visuals, exactly as the API produces them

In [ ]:
record = storage.create_result(
    "ndvi",
    title="NDVI analysis - notebook 08 demo",
    dataset=metadata["name"],
    aoi=aoi["name"],
    params={"index": "NDVI", "bands": resolution["bands"], "thresholds": thresholds, "notebook": "08_final_demo"},
    cache_key=None,
)
output_dir = storage.result_directory(record["id"], create=True)

assets = visualization.render_result_bundle(
    output_dir,
    rgb=preprocessing.composite_rgb({key: stack["bands"][key] for key in ("red", "green", "blue")}, valid),
    index_array=ndvi,
    classes=classes,
    weak_mask=np.asarray(weak["mask"]) if weak["mask"].any() else None,
    weak_threshold=weak["threshold"],
    weak_patch_count=weak["patch_count"],
    statistics=statistics,
    health=health,
    transform=stack["transform"],
    crs=stack["crs"],
    aoi_gdf=aoi["gdf"],
    bounds=clipped["clip_bounds"],
    valid=valid,
    index_name="NDVI",
    dataset_label=metadata["name"],
    extra_caption="notebook 08 · %d valid pixels" % statistics["valid_pixels"],
)

for key, item in assets.items():
    storage.add_asset(
        record,
        item["path"],
        name=key,
        kind="overlay" if item.get("kind") == "overlay" else item.get("kind", "image"),
        label=item.get("label", key),
        bounds_lonlat=item.get("bounds_lonlat"),
        extra={"map_key": key},
    )
    print(f"  {key:22s} {item.get('kind', 'image'):8s} {pathlib.Path(item['path']).name}")

## Step 5 · Store the result so the dashboard and `/api/results` can see it

In [ ]:
observation_date = metadata["tags"].get("ACQUIRED") or "1970-01-01"
payload = {
    "result_id": record["id"],
    "kind": "ndvi",
    "index": "NDVI",
    "dataset": metadata["name"],
    "aoi": aoi["name"],
    "observation_date": observation_date,
    "statistics": statistics,
    "health": health,
    "weak_vegetation": {key: value for key, value in weak.items() if key != "mask"},
    "anomalies": anomalies,
    "zones": {"total_zones": zones["total_zones"], "zone_count": zones["zone_count"]},
    "field": summary,
    "interpretation": interpretation,
    "assets": record["assets"],
}

warnings = list(validation["warnings"]) + list(aoi["warnings"])
if metadata["synthetic"]:
    warnings.append("This dataset is a labelled synthetic demonstration scene, not real satellite or drone imagery.")

record["payload"] = storage.jsonable(payload)
record["summary"] = {
    "index": "NDVI",
    "mean": statistics["mean"],
    "healthy_pct": health["healthy_pct"],
    "weak_pct": health["weak_pct"],
    "observation_date": observation_date,
}
record["warnings"] = warnings
record["observation_date"] = observation_date
storage.save_result(record)

print("result id  :", record["id"])
print("stored in  :", storage.result_directory(record["id"]).relative_to(ROOT))
print("assets     :", len(record["assets"]))
print("warnings   :")
for warning in warnings:
    print("  -", warning)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].imshow(np.where(valid & np.isfinite(ndvi), ndvi, np.nan), cmap="RdYlGn", vmin=-0.2, vmax=1.0)
axes[0].set_title("NDVI - %s" % metadata["name"])
axes[0].axis("off")
axes[1].imshow(plt.imread(pathlib.Path(assets["health_map"]["path"])))
axes[1].set_title("Crop-health map")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("Interpretation:", interpretation["verdict"])
for point in interpretation["points"]:
    print(" -", point)

## Step 6 · What just happened, and what comes next

In [ ]:
print("Results now in the library:")
for entry in storage.list_results(limit=4):
    print(" ", entry["id"], "|", entry["kind"], "|", entry["dataset"], "|", len(entry["assets"]), "assets")

print("\nThe same pipeline is available as HTTP:")
for endpoint in ("POST /api/ndvi", "POST /api/crop-health", "POST /api/segmentation", "POST /api/timeseries", "GET /api/results/{id}"):
    print("  ", endpoint)

### Next steps for a real field POC

1. Replace the demo scene: drop a Sentinel-2 L2A (or drone) GeoTIFF plus your
   field boundary GeoJSON into `data/raw`, then point `RASTER`/`AOI_FILE` at
   them. If the file has no band descriptions, pass an explicit band map.
2. Confirm the band mapping before trusting absolute values - the engine tells
   you when it assumed one.
3. Keep the acquisition date: it drives the time-series page.
4. Repeat per date, then run notebook 07 for the trajectory.
5. Calibrate `crop_health.thresholds` for your crop and sensor in `config.yaml`.

**A single NDVI snapshot ranks in-field variability. It does not diagnose
disease.** Treat weak patches as scouting targets and confirm on the ground.